IMPORTS 

In [ ]:
import numpy as np
import pandas as pd
import os, math, time
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing  import StandardScaler, LabelEncoder
from sklearn.linear_model   import LinearRegression, LogisticRegression
from sklearn.ensemble       import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics        import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
from scipy.stats import randint, uniform
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

print('Libraries loaded successfully.')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device    : {DEVICE}')


Libraries loaded successfully.
PyTorch version : 2.8.0+cpu
CUDA available  : False
Using device    : cpu


DATA LOADING

In [3]:
DATA_PATH = 'cleaned_taxi_data.parquet'

def load_data(path : str) -> pd.DataFrame:
    if path.endswith('.parquet'):
        return pd.read_parquet(path)
    return pd.read_csv(path, low_memory=False)

df_raw = load_data(DATA_PATH)
print(f'Raw dataset shape : {df_raw.shape}')
print(f'Columns           : {df_raw.columns.tolist()}')

Raw dataset shape : (2241617, 19)
Columns           : ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee']


In [4]:

df = df_raw[df_raw['payment_type'] == 1].copy()
print(f'After credit-card filter : {df.shape[0]:,} rows  ({df.shape[0]/df_raw.shape[0]:.1%} of original)')

for col in ['tpep_pickup_datetime', 'tpep_dropoff_datetime']:
    if col in df.columns and not pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = pd.to_datetime(df[col])

df.head(3)

After credit-card filter : 2,241,617 rows  (100.0% of original)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.8,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.7,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
2,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.4,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0


In [5]:
ZONE_LOOKUP_PATH = 'taxi_zone_lookup.csv'

zone_df = pd.read_csv(ZONE_LOOKUP_PATH)
print(f'Zone lookup shape : {zone_df.shape}')
loc_to_borough = zone_df.set_index('LocationID')['Borough'].to_dict()
print('Zone lookup loaded. Sample:', dict(list(loc_to_borough.items())[:4]))


Zone lookup shape : (265, 4)
Zone lookup loaded. Sample: {1: 'EWR', 2: 'Queens', 3: 'Bronx', 4: 'Manhattan'}


FEATURE ENGINEERING

In [8]:
def engineer_features(df : pd.DataFrame, loc_to_borough : dict) -> pd.DataFrame:
    df = df.copy()

    df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
    df['pickup_day_of_week'] = df['tpep_pickup_datetime'].dt.dayofweek
    df['is_weekend'] = (df['pickup_day_of_week'] >= 5).astype(int)
  
    df['trip_duration_minutes'] = (
        (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
    )

    df['trip_speed_mph'] = np.where(
        df['trip_duration_minutes'] > 0,
        (df['trip_distance'] / (df['trip_duration_minutes'] / 60)),
        np.nan 
    )

    df['log_trip_distance'] = np.log1p(df['trip_distance'])

    df['fare_per_mile'] = np.where(
        df['trip_distance'] > 0,
        df['fare_amount'] / df['trip_distance'],
        np.nan 
    )

    df['fare_per_minute'] = np.where(
        df['trip_duration_minutes'] > 0,
        df['fare_amount'] / df['trip_duration_minutes'],
        np.nan 
    )

    df['pickup_borough'] = df['PULocationID'].map(loc_to_borough).fillna('Unknown')
    df['dropoff_borough'] = df['DOLocationID'].map(loc_to_borough).fillna('Unknown')

    le = LabelEncoder()
    df['pickup_borough_enc'] = le.fit_transform(df['pickup_borough'])
    df['dropoff_borough_enc'] = le.fit_transform(df['dropoff_borough'])

    return df

df = engineer_features(df, loc_to_borough)
print('Feature engineering finished.')
print(df[['pickup_hour', 'pickup_day_of_week', 'is_weekend', 
          'trip_duration_minutes', 'trip_speed_mph', 'log_trip_distance',
          'fare_per_mile', 'fare_per_minute', 'pickup_borough', 'dropoff_borough']].describe().round(2)  )

Feature engineering finished.
       pickup_hour  pickup_day_of_week  is_weekend  trip_duration_minutes  \
count   2241617.00          2241617.00  2241617.00             2241617.00   
mean         14.37                2.88        0.26                  14.62   
std           5.67                1.93        0.44                  11.01   
min           0.00                0.00        0.00                   1.00   
25%          11.00                1.00        0.00                   7.27   
50%          15.00                3.00        0.00                  11.60   
75%          19.00                5.00        1.00                  18.43   
max          23.00                6.00        1.00                 294.00   

       trip_speed_mph  log_trip_distance  fare_per_mile  fare_per_minute  
count      2241617.00         2241617.00     2241617.00       2241617.00  
mean            11.51               1.17           7.85             1.27  
std             22.69               0.65          2